In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

processed_dir = '../data/processed/'
df = pd.read_csv(os.path.join(processed_dir, 'featured_orders.csv'))

print("Data loaded successfully! Shape:", df.shape)
print("Columns available:", df.columns.tolist())

Data loaded successfully! Shape: (20639, 15)
Columns available: ['orderid', 'customerid', 'restaurantid', 'deliverypartnerid', 'orderdate', 'ordertime', 'deliverytimeminutes', 'foodcost', 'deliveryfee', 'discount', 'couponcode', 'gst', 'finalamount', 'orderstatus', 'paymentmethod']


In [2]:
target_col = 'churn' # Change this if your column name is different (e.g., 'is_churn')

if target_col not in df.columns and 'is_churn' in df.columns:
    target_col = 'is_churn'

if target_col in df.columns:
    X = df.drop(columns=[target_col, 'orderid', 'orderdate', 'ordertime'], errors='ignore')
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print("Data split into training and testing sets successfully.")
else:
    print(f"Target column '{target_col}' not found in dataset. Please verify column names.")

Target column 'churn' not found in dataset. Please verify column names.


In [3]:
print(df.columns.tolist())

['orderid', 'customerid', 'restaurantid', 'deliverypartnerid', 'orderdate', 'ordertime', 'deliverytimeminutes', 'foodcost', 'deliveryfee', 'discount', 'couponcode', 'gst', 'finalamount', 'orderstatus', 'paymentmethod']


In [4]:
# orderstatus के आधार पर churn target column बनाएं (Cancelled होने पर 1, वरना 0)
if 'orderstatus' in df.columns:
    df['churn'] = (df['orderstatus'] == 'Cancelled').astype(int)

target_col = 'churn'

if target_col in df.columns:
    # गैर-जरूरी आईडी, डेट और टेक्स्ट कॉलम्स को ड्रॉप करें
    drop_cols = [target_col, 'orderid', 'customerid', 'restaurantid', 'deliverypartnerid', 'orderdate', 'ordertime', 'orderstatus']
    X = df.drop(columns=drop_cols, errors='ignore')
    
    # Categorical columns (जैसे couponcode, paymentmethod) को numeric में बदलें
    X = pd.get_dummies(X, drop_first=True)
    
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print("Data split into training and testing sets successfully. Shape of X:", X.shape)
else:
    print("Target column could not be created.")

Data split into training and testing sets successfully. Shape of X: (20639, 8)


In [5]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"--- {name} Performance ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("-" * 50)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)

In [6]:
# orderstatus के आधार पर churn target column बनाएं
if 'orderstatus' in df.columns:
    df['churn'] = (df['orderstatus'] == 'Cancelled').astype(int)

target_col = 'churn'

if target_col in df.columns:
    print("Unique classes in target:", df[target_col].unique())
    print("Value counts:\n", df[target_col].value_counts())
    
    drop_cols = [target_col, 'orderid', 'customerid', 'restaurantid', 'deliverypartnerid', 'orderdate', 'ordertime', 'orderstatus']
    X = df.drop(columns=drop_cols, errors='ignore')
    X = pd.get_dummies(X, drop_first=True)
    y = df[target_col]
    
    # Check if we have at least 2 classes
    if y.nunique() > 1:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        print("Data split into training and testing sets successfully.")
    else:
        print("Error: Target column contains only one class. Please check your data or choose a different target condition.")
else:
    print("Target column could not be created.")

Unique classes in target: [0]
Value counts:
 churn
0    20639
Name: count, dtype: int64
Error: Target column contains only one class. Please check your data or choose a different target condition.


In [7]:
print(df['orderstatus'].value_counts())

orderstatus
-0.127907    11480
-1.262363     4606
 2.141005     2380
 1.006549     2173
Name: count, dtype: int64


In [8]:
# orderstatus की सबसे कम फ्रीक्वेंसी वाली वैल्यू को churn (1) और बाकी को 0 सेट करें
target_value = df['orderstatus'].value_counts().index[-1]
df['churn'] = (df['orderstatus'] == target_value).astype(int)

target_col = 'churn'
print("Value counts for churn:\n", df[target_col].value_counts())

drop_cols = [target_col, 'orderid', 'customerid', 'restaurantid', 'deliverypartnerid', 'orderdate', 'ordertime', 'orderstatus']
X = df.drop(columns=drop_cols, errors='ignore')
X = pd.get_dummies(X, drop_first=True)
y = df[target_col]

if y.nunique() > 1:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print("Data split into training and testing sets successfully.")
else:
    print("Error: Target column contains only one class.")

Value counts for churn:
 churn
0    18466
1     2173
Name: count, dtype: int64
Data split into training and testing sets successfully.


In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"--- {name} Performance ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("-" * 50)

--- Logistic Regression Performance ---
Accuracy: 0.8934108527131783
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.99      0.94      3693
           1       0.45      0.06      0.10       435

    accuracy                           0.89      4128
   macro avg       0.68      0.52      0.52      4128
weighted avg       0.85      0.89      0.85      4128

--------------------------------------------------
--- Random Forest Classifier Performance ---
Accuracy: 0.9527616279069767
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.99      0.97      3693
           1       0.84      0.68      0.75       435

    accuracy                           0.95      4128
   macro avg       0.90      0.83      0.86      4128
weighted avg       0.95      0.95      0.95      4128

--------------------------------------------------


In [10]:
os.makedirs('../models', exist_ok=True)

best_churn_model = models["Random Forest Classifier"]
joblib.dump(best_churn_model, '../models/churn_model.pkl')
print("Best churn model successfully saved to '../models/churn_model.pkl'!")

Best churn model successfully saved to '../models/churn_model.pkl'!
